In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')

try:
    import scvi
except ImportError:
    raise ImportError("scvi-tools not installed. Run: pip install scvi-tools")

import torch

# Detect best available accelerator (Apple Silicon MPS / CUDA GPU / CPU)
if torch.backends.mps.is_available():
    accelerator = "mps"
elif torch.cuda.is_available():
    accelerator = "gpu"
else:
    accelerator = "cpu"

# Reproducibility — fix seeds for scVI and numpy
scvi.settings.seed = 42
np.random.seed(42)
sc.settings.verbosity = 1

from pathlib import Path
base_path = os.environ.get("TRAJRISK_ROOT") or str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "metadata" / "source_provenance.json").is_file()))
data_processed = os.path.join(base_path, "Data/processed")
results_figs   = os.path.join(base_path, "results/figures")
models_path    = os.path.join(base_path, "models")

os.makedirs(models_path, exist_ok=True)
os.makedirs(results_figs, exist_ok=True)

print(f"scanpy:      {sc.__version__}")
print(f"scvi-tools:  {scvi.__version__}")
print(f"torch:       {torch.__version__}")
print(f"accelerator: {accelerator}")

In [ ]:
# We load from the pre-normalization files because scVI requires raw counts.
# The *_normalized.h5ad files have log-transformed X — not suitable for scVI.

print("=" * 80)
print("Load mouse expression cohorts")
print("=" * 80)

raw_file_map = {
    'Walesky':        'walesky_apap_control_raw.h5ad',
    'Umbaugh_2h':     'umbaugh_2h_filtered.h5ad',
    'Ben_Moshe':      'benmoshe_hepatocytes.h5ad',
    'Umbaugh_48h72h': 'umbaugh_48h72h_hepatocytes.h5ad',
}

# Exclude per-cohort QC annotations from integration metadata.
qc_cols = {
    'n_genes_by_counts', 'total_counts', 'pct_counts_mt',
    'log1p_total_counts', 'log1p_n_genes_by_counts',
    'total_counts_mt', 'UMAP_X', 'UMAP_Y', 'leiden',
    'pct_counts_in_top_50_genes', 'n_genes', 'mt'
}

adatas_raw = {}

for name, fname in raw_file_map.items():
    fpath = os.path.join(data_processed, fname)
    adata = sc.read_h5ad(fpath)
    adatas_raw[name] = adata

    # Integer-count check on a sample of non-zero values
    if hasattr(adata.X, 'data') and len(adata.X.data) > 0:
        sample_vals = adata.X.data[:500]
    else:
        arr = (adata.X[:30, :30].toarray()
               if hasattr(adata.X, 'toarray') else np.array(adata.X[:30, :30]))
        sample_vals = arr.flatten()[arr.flatten() > 0]

    is_int = (bool(np.all(sample_vals == np.floor(sample_vals)))
              if len(sample_vals) > 0 else False)

    useful_cols = [c for c in adata.obs.columns if c not in qc_cols]

    print(f"\n{name}:")
    print(f"  Shape:          {adata.n_obs:,} cells × {adata.n_vars:,} genes")
    print(f"  X dtype:        {adata.X.dtype}")
    print(f"  Integer counts: {'YES ✅' if is_int else 'NO ⚠️  (pre-processed)'}")
    for col in useful_cols:
        uniq = sorted(adata.obs[col].dropna().astype(str).unique().tolist())
        print(f"  .obs['{col}']: {uniq[:8]}{'...' if len(uniq) > 8 else ''}")

In [ ]:
print("=" * 80)
print("HARMONIZING METADATA")
print("=" * 80)

def harmonize_metadata(adatas_raw):
    out = {}

    for name, adata in adatas_raw.items():
        adata = adata.copy()
        adata.obs['dataset'] = name

        # WALESKY
        # 'treatment' and 'timepoint' columns confirmed present from Cell 2.
        # 'timepoint_hours' exists but is stored as strings ('0', '6', etc.).
        if name == 'Walesky':
            adata.obs['treatment']       = adata.obs['treatment'].astype(str).str.strip()
            adata.obs['timepoint_str']   = adata.obs['timepoint'].astype(str).str.strip()
            # Ensure 'h' suffix
            adata.obs['timepoint_str'] = adata.obs['timepoint_str'].apply(
                lambda x: x if x.endswith('h') else x + 'h'
            )
            adata.obs['timepoint_hours'] = adata.obs['timepoint_hours'].astype(float)

        # UMBAUGH 2H
        # 'timepoint' is '2h' for ALL cells — including Control — because samples
        # were harvested at the same endpoint. Remap to biological time:
        # Control = t0 (no injury), APAP = t2h (early injury).
        elif name == 'Umbaugh_2h':
            adata.obs['treatment']     = adata.obs['treatment'].astype(str).str.strip()
            tp_str = {'Control': '0h', 'APAP': '2h'}
            tp_hr  = {'Control': 0.0,  'APAP': 2.0}
            adata.obs['timepoint_str']   = adata.obs['treatment'].map(tp_str)
            adata.obs['timepoint_hours'] = adata.obs['treatment'].map(tp_hr)

        # BEN-MOSHE
        # 'time_point' confirmed as integer strings: '0', '48', '72'.
        # No explicit treatment column — infer from time_point.
        elif name == 'Ben_Moshe':
            adata.obs['timepoint_hours'] = adata.obs['time_point'].astype(int).astype(float)
            adata.obs['timepoint_str']   = adata.obs['time_point'].astype(int).astype(str) + 'h'
            adata.obs['treatment']       = np.where(
                adata.obs['timepoint_hours'] == 0, 'Control', 'APAP'
            )

        # UMBAUGH 48H/72H
        # 'treatment' and 'timepoint' confirmed present. timepoint_hours exists
        # but may be stored as strings — cast to float.
        elif name == 'Umbaugh_48h72h':
            adata.obs['treatment']     = adata.obs['treatment'].astype(str).str.strip()
            adata.obs['timepoint_str'] = adata.obs['timepoint'].astype(str).str.strip()
            adata.obs['timepoint_str'] = adata.obs['timepoint_str'].apply(
                lambda x: x if x.endswith('h') else x + 'h'
            )
            hr_map = {'48h': 48.0, '72h': 72.0}
            adata.obs['timepoint_hours'] = adata.obs['timepoint_str'].map(hr_map)

        # Standardize treatment casing
        adata.obs['treatment'] = (
            adata.obs['treatment'].astype(str).str.strip()
            .replace({'control': 'Control', 'apap': 'APAP'})
        )

        # Verify all required columns present and populated
        required = ['dataset', 'treatment', 'timepoint_str', 'timepoint_hours']
        missing  = [
            c for c in required
            if c not in adata.obs.columns or adata.obs[c].isna().all()
        ]

        print(f"\n{name}:")
        if missing:
            print(f"   MISSING: {missing}")
        else:
            print(f"  treatment:       {sorted(adata.obs['treatment'].unique())}")
            print(f"  timepoint_str:   {sorted(adata.obs['timepoint_str'].unique())}")
            print(f"  timepoint_hours: {sorted(adata.obs['timepoint_hours'].dropna().unique())}")

        out[name] = adata

    return out

adatas_harmonized = harmonize_metadata(adatas_raw)

In [ ]:
print("=" * 80)
print("INCLUSION DECISION")
print("=" * 80)

def has_integer_counts(adata):
    if hasattr(adata.X, 'data') and len(adata.X.data) > 0:
        vals = adata.X.data[:1000]
    else:
        arr = (adata.X[:30, :30].toarray()
               if hasattr(adata.X, 'toarray') else np.array(adata.X[:30, :30]))
        vals = arr.flatten()[arr.flatten() > 0]
    return bool(np.all(vals == np.floor(vals))) if len(vals) > 0 else False

any_non_integer = False

for name, adata in adatas_harmonized.items():
    is_int = has_integer_counts(adata)
    if not is_int:
        any_non_integer = True
    status = "integer counts" if is_int else "pre-processed (float)"
    print(f"  {name}: {status}")

print()
if any_non_integer:
    print("  Non-integer values detected in one or more datasets.")
    print("   All four datasets included using scVI with gene_likelihood='normal'.")
    LIKELIHOOD = 'normal'
else:
    print(" All datasets have integer counts — using gene_likelihood='nb'.")
    LIKELIHOOD = 'nb'

print(f"\n   Selected likelihood: '{LIKELIHOOD}'")

# All four datasets included in integration
for_scvi = adatas_harmonized

print(f"   Datasets for scVI: {list(for_scvi.keys())}")
print(f"   Total cells: {sum(a.n_obs for a in for_scvi.values()):,}")

In [ ]:
print("=" * 80)
print("CONCATENATING MOUSE DATASETS")
print("=" * 80)

required_cols = ['dataset', 'treatment', 'timepoint_str', 'timepoint_hours']

adatas_list = []
keys_list   = []

for name, adata in for_scvi.items():
    # Drop cells with any missing required metadata
    mask = adata.obs[required_cols].notna().all(axis=1)
    n_dropped = (~mask).sum()
    if n_dropped > 0:
        print(f"  {name}: dropping {n_dropped} cells with incomplete metadata")
    adatas_list.append(adata[mask].copy())
    keys_list.append(name)

# join='inner' keeps only genes present in every dataset
adata_combined = sc.concat(
    adatas_list,
    join='inner',
    merge='same',
    label='dataset_concat',
    keys=keys_list,
    index_unique='-',   # avoids duplicate cell barcodes across datasets
)

# Remove the redundant auto-generated dataset_concat column
if 'dataset_concat' in adata_combined.obs.columns:
    del adata_combined.obs['dataset_concat']

adata_combined.var_names_make_unique()

print(f"\nCombined object:")
print(f"  Cells: {adata_combined.n_obs:,}")
print(f"  Genes: {adata_combined.n_vars:,} (intersection across all included datasets)")
print(f"\n  Cells per dataset:")
for ds, cnt in adata_combined.obs['dataset'].value_counts().items():
    print(f"    {ds}: {cnt:,}")
print(f"\n  Cells per treatment:")
for t, cnt in adata_combined.obs['treatment'].value_counts().items():
    print(f"    {t}: {cnt:,}")
print(f"\n  Timepoints present: {sorted(adata_combined.obs['timepoint_str'].unique())}")

In [ ]:
# For gene_likelihood='normal', scVI models log-normalized expression directly.
# No raw counts layer is needed. We log-normalize all datasets to a common scale
# (target_sum=1e4, log1p) before integration. For datasets already carrying
# some degree of normalization (Walesky, Ben-Moshe), this brings them onto a
# consistent scale with the raw-count datasets.

print("=" * 80)
print("PREPARING FOR scVI (Normal likelihood)")
print("=" * 80)

# Log-normalize: handles both raw counts and pre-processed values
print("Log-normalizing combined dataset...")
sc.pp.normalize_total(adata_combined, target_sum=1e4)
sc.pp.log1p(adata_combined)

# Batch-aware HVG selection
# Using batch_key='dataset' prevents selection of genes that are
# variable in only one dataset due to platform-specific effects.
print("Selecting batch-aware highly variable genes...")
sc.pp.highly_variable_genes(
    adata_combined,
    n_top_genes=4000,
    batch_key='dataset',
    flavor='seurat',
    subset=False,
)

n_hvg = adata_combined.var['highly_variable'].sum()
print(f"Batch-aware HVGs selected: {n_hvg} / {adata_combined.n_vars}")

# Subset to HVGs for scVI input
adata_scvi = adata_combined[:, adata_combined.var['highly_variable']].copy()

print(f"\nscVI input: {adata_scvi.n_obs:,} cells × {adata_scvi.n_vars:,} HVGs")
print(f"Value range: min={np.min(adata_scvi.X):.3f}, max={np.max(adata_scvi.X):.3f}")
print(f"Likelihood to use: '{LIKELIHOOD}'")

In [ ]:
# batch_key='dataset' corrects for dataset-level technical variation only.
# Treatment and timepoint are preserved as biological variables for
# downstream trajectory inference and must NOT be included as covariates.

print("=" * 80)
print("scVI SETUP AND TRAINING")
print("=" * 80)

# For Normal likelihood, scVI reads directly from X (log-normalized)
# No layer argument needed.
scvi.model.SCVI.setup_anndata(
    adata_scvi,
    batch_key='dataset',
)

model = scvi.model.SCVI(
    adata_scvi,
    n_latent=30,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood=LIKELIHOOD,  # 'normal' for mixed data, 'nb' for raw counts
    dispersion='gene',
)

print(model)
print(f"\nTraining accelerator: {accelerator.upper()}")
print(f"Total parameters: {sum(p.numel() for p in model.module.parameters()):,}")
print("\nTraining — monitor ELBO_validation for convergence...\n")

model.train(
    max_epochs=400,
    early_stopping=True,
    early_stopping_patience=20,
    train_size=0.9,
    batch_size=128,
    accelerator=accelerator,
)

model_dir = os.path.join(models_path, "scvi_trajrisk_model")
model.save(model_dir, overwrite=True)
print(f"\n Model saved: {model_dir}")

# Training history
train_elbo = model.history['elbo_train']
val_elbo   = model.history['elbo_validation']

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_elbo.index, train_elbo.values, label='Train ELBO', color='#2196F3')
ax.plot(val_elbo.index,   val_elbo.values,   label='Val ELBO',   color='#FF5722')
ax.set_xlabel('Epoch')
ax.set_ylabel('ELBO')
ax.set_title('scVI Training History')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(results_figs, 'scvi_training_history.png'), dpi=300, bbox_inches='tight')
plt.show()
print(" Training history saved")

In [ ]:
print("=" * 80)
print("EXTRACTING LATENT REPRESENTATION")
print("=" * 80)

# The latent representation is a 30-dim embedding that accounts for
# dataset-level batch effects while preserving biological variation.
adata_scvi.obsm['X_scvi'] = model.get_latent_representation()

print(f"Latent space shape: {adata_scvi.obsm['X_scvi'].shape}")

# Build neighborhood graph on the scVI latent space — NOT on PCA.
# n_pcs=None is explicit: we are using X_scvi, not any PCA reduction.
sc.pp.neighbors(
    adata_scvi,
    use_rep='X_scvi',
    n_neighbors=15,
    n_pcs=None,
)

sc.tl.umap(adata_scvi, min_dist=0.3)
sc.tl.leiden(adata_scvi, resolution=0.5, key_added='leiden_scvi')

n_clusters = adata_scvi.obs['leiden_scvi'].nunique()
print(f"Leiden clusters on scVI latent space: {n_clusters}")
print(" UMAP and clustering complete")

In [ ]:
# Set correct categorical order for timepoint_str
# so UMAP legends display 0h → 2h → 6h → 24h → 48h → 72h → 96h

correct_order = ['0h', '2h', '6h', '24h', '48h', '72h', '96h']

adata_scvi.obs['timepoint_str'] = pd.Categorical(
    adata_scvi.obs['timepoint_str'],
    categories=correct_order,
    ordered=True
)

print(" Timepoint order set:")
print(f"   {list(adata_scvi.obs['timepoint_str'].cat.categories)}")

In [ ]:
# Expected outcome:
#   dataset UMAP  → datasets should be MIXED (no isolated islands per dataset)
#   timepoint UMAP → temporal gradient 0h → 96h should be visible
#   treatment UMAP → APAP should separate from Control
#   markers        → stress genes (Hmox1) should increase along injury axis

mouse_markers = ['Hmox1', 'Alb', 'Cyp2e1', 'Gsta1']
markers_in_data = [m for m in mouse_markers if m in adata_scvi.var_names]

n_marker_plots = len(markers_in_data)
total_plots    = 5 + n_marker_plots
n_cols         = 3
n_rows         = int(np.ceil(total_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten()
idx  = 0

sc.pl.umap(adata_scvi, color='dataset',
           ax=axes[idx], show=False, frameon=False,
           title='Dataset — should mix well across batches')
idx += 1

sc.pl.umap(adata_scvi, color='treatment',
           ax=axes[idx], show=False, frameon=False,
           title='Treatment')
idx += 1

sc.pl.umap(adata_scvi, color='timepoint_hours',
           ax=axes[idx], show=False, frameon=False,
           color_map='viridis',
           title='Timepoint (hours post-APAP)')
idx += 1

sc.pl.umap(adata_scvi, color='timepoint_str',
           ax=axes[idx], show=False, frameon=False,
           title='Timepoint (labeled)')
idx += 1

sc.pl.umap(adata_scvi, color='leiden_scvi',
           ax=axes[idx], show=False, frameon=False,
           title='Leiden clusters (scVI latent space)')
idx += 1

for marker in markers_in_data:
    if idx < len(axes):
        sc.pl.umap(adata_scvi, color=marker,
                   ax=axes[idx], show=False, frameon=False,
                   color_map='Reds', use_raw=False,
                   title=marker)
        idx += 1

for i in range(idx, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
fig_path = os.path.join(results_figs, 'scvi_integration_umap.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f" Figure saved: {fig_path}")

In [ ]:
from sklearn.metrics import silhouette_score
from scipy.stats    import spearmanr

print("=" * 80)
print("QUANTITATIVE INTEGRATION VALIDATION")
print("=" * 80)

latent      = adata_scvi.obsm['X_scvi']
sample_size = min(5000, adata_scvi.n_obs)

# 1. Batch mixing: silhouette of dataset labels in latent space.
#    Closer to 0 = datasets are well mixed. Positive = still separated by batch.
sil_batch = silhouette_score(
    latent, adata_scvi.obs['dataset'],
    sample_size=sample_size, random_state=42
)

# 2. Biological conservation: silhouette of treatment in latent space.
#    Should be positive — APAP-treated cells ought to be distinguishable.
sil_treatment = silhouette_score(
    latent, adata_scvi.obs['treatment'],
    sample_size=sample_size, random_state=42
)

# 3. Temporal ordering: Spearman correlation between timepoint_hours and UMAP-1.
#    A strong correlation indicates temporal structure is preserved in the embedding.
rho, pval = spearmanr(
    adata_scvi.obs['timepoint_hours'].fillna(0),
    adata_scvi.obsm['X_umap'][:, 0]
)

print(f"\n  Batch mixing ASW (dataset labels):  {sil_batch:.4f}")
print(f"    → Target: close to 0 (datasets intermix in latent space)")

print(f"\n  Biological ASW (treatment labels):  {sil_treatment:.4f}")
print(f"    → Target: > 0.05 (APAP vs Control remain distinguishable)")

print(f"\n  Temporal ordering (Spearman ρ):     {rho:.4f}  (p = {pval:.2e})")
print(f"    → Target: |ρ| > 0.30 (timepoints ordered along embedding)")

print(f"\n{'─'*60}")
if sil_batch > 0.3:
    print("    BATCH NOT MIXING — increase n_latent to 50, re-train")
elif sil_treatment < 0.0:
    print("    BIOLOGICAL SIGNAL LOST — decrease n_latent to 20, re-train")
elif abs(rho) < 0.15:
    print("    Weak temporal correlation; inspect trajectory-root selection and integration diagnostics.")
else:
    print("   Integration acceptable — proceed to trajectory inference")
print(f"{'─'*60}")

In [ ]:
# Check temporal correlation across BOTH UMAP axes

from scipy.stats import spearmanr

tp_hours = adata_scvi.obs['timepoint_hours'].fillna(0)

rho1, p1 = spearmanr(tp_hours, adata_scvi.obsm['X_umap'][:, 0])
rho2, p2 = spearmanr(tp_hours, adata_scvi.obsm['X_umap'][:, 1])

print(f"UMAP-1 vs timepoint_hours: ρ = {rho1:.4f}, p = {p1:.2e}")
print(f"UMAP-2 vs timepoint_hours: ρ = {rho2:.4f}, p = {p2:.2e}")
print(f"\nMax |ρ| across axes: {max(abs(rho1), abs(rho2)):.4f}")

In [ ]:
# Check if temporal signal exists in the 30D scVI latent space
# UMAP is lossy — the real question is what's in X_scvi directly

from sklearn.decomposition import PCA
from scipy.stats import spearmanr

tp_hours = adata_scvi.obs['timepoint_hours'].fillna(0)

# Project 30D latent space into 10 PCs
pca_latent = PCA(n_components=10, random_state=42).fit_transform(
    adata_scvi.obsm['X_scvi']
)

print("Spearman ρ: timepoint_hours vs PCs of 30D scVI latent space")
print("=" * 60)

best_rho = 0
for i in range(10):
    rho, p = spearmanr(tp_hours, pca_latent[:, i])
    flag = "✅" if abs(rho) > 0.30 else "⚠️ "
    print(f"  PC{i+1:02d}: ρ = {rho:+.4f}  p = {p:.2e}  {flag}")
    if abs(rho) > abs(best_rho):
        best_rho = rho

print(f"\n  Best |ρ| in 30D latent space: {abs(best_rho):.4f}")
print("=" * 60)

if abs(best_rho) >= 0.30:
    print("   Temporal ordering IS in the latent space — UMAP was just lossy")
    print("     Safe to proceed to trajectory inference")
elif abs(best_rho) >= 0.15:
    print("    Weak temporal ordering in latent space")
    print("     Monocle 3 may still work — proceed with caution")
else:
    print("   Temporal signal largely absent from latent space")
    print("     Consider re-training with n_latent=50 or n_latent=20")

In [ ]:
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
import numpy as np

tp_hours = adata_scvi.obs['timepoint_hours'].fillna(0).values.reshape(-1, 1)

# How much temporal variance do ALL 10 PCs explain together?
pca_latent = PCA(n_components=10, random_state=42).fit_transform(
    adata_scvi.obsm['X_scvi']
)

reg = LinearRegression().fit(pca_latent, tp_hours)
r2_combined = reg.score(pca_latent, tp_hours)

print(f"Combined R² (10 PCs → timepoint_hours): {r2_combined:.4f}")
print(f"Combined R (correlation):                {np.sqrt(r2_combined):.4f}")

if r2_combined > 0.15:
    print("\n Combined temporal signal is strong — safe to proceed")
elif r2_combined > 0.08:
    print("\n  Combined temporal signal is moderate — proceed with caution")
else:
    print("\n Temporal signal too weak — consider re-training")

In [ ]:
print("=" * 80)
print("SAVING INTEGRATED DATASET")
print("=" * 80)

# Transfer scVI embeddings back to adata_combined (which retains all genes).
# Index alignment via obs_names ensures correct cell-to-embedding mapping.
adata_combined.obsm['X_scvi'] = adata_scvi.obsm['X_scvi']
adata_combined.obsm['X_umap'] = adata_scvi.obsm['X_umap']

adata_combined.obs.loc[adata_scvi.obs_names, 'leiden_scvi'] = (
    adata_scvi.obs['leiden_scvi'].values
)

output_path = os.path.join(data_processed, "adata_integrated.h5ad")
adata_combined.write_h5ad(output_path)

size_mb = os.path.getsize(output_path) / 1e6
print(f" Saved: {output_path}")
print(f"   Size:          {size_mb:.1f} MB")
print(f"   Cells:         {adata_combined.n_obs:,}")
print(f"   Genes:         {adata_combined.n_vars:,}")
print(f"   Latent space:  {adata_combined.obsm['X_scvi'].shape}")
print(f"   Model:         {os.path.join(models_path, 'scvi_trajrisk_model')}")

print("\nFiles created this week:")
print(f"  {output_path}")
print(f"  {os.path.join(models_path, 'scvi_trajrisk_model')}")
print(f"  {os.path.join(results_figs, 'scvi_training_history.png')}")
print(f"  {os.path.join(results_figs, 'scvi_integration_umap.png')}")